In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# Path สำหรับบันทึกไฟล์
save_path = r"C:\Users\n_noo\Desktop\research\Piyawat-Project\data\raw\wunderground_weather_VTPT_2014_2026.csv"

# โหลดไฟล์เดิมถ้ามีอยู่แล้ว
if os.path.exists(save_path):
    all_data = pd.read_csv(save_path, encoding="utf-8-sig")
    print("โหลดข้อมูลเดิมเรียบร้อย ✅")
else:
    all_data = pd.DataFrame()

# ฟังก์ชันดึงข้อมูลจาก Wunderground
def get_weather_data(year, month):
    url = f"https://www.wunderground.com/history/monthly/th/mueang-tak/VTPT/date/{year}-{month}"
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table")
        if table:
            df = pd.read_html(str(table))[0]
            df["Year"] = year
            df["Month"] = month
            return df
    return None

# ลูปตั้งแต่ปี 2014 ถึง 2026
for year in range(2014, 2027):
    for month in range(1, 13):
        # ตรวจสอบว่ามีข้อมูลนี้แล้วหรือยัง
        if ((all_data["Year"] == year) & (all_data["Month"] == month)).any():
            print(f"ข้าม {year}-{month} (มีข้อมูลแล้ว)")
            continue

        print(f"กำลังดึงข้อมูล {year}-{month} ...")
        df = get_weather_data(year, month)
        if df is not None:
            all_data = pd.concat([all_data, df], ignore_index=True)
            # บันทึกทันทีหลังดึงแต่ละเดือน เพื่อกันข้อมูลหาย
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            all_data.to_csv(save_path, index=False, encoding="utf-8-sig")
        time.sleep(1)  # หน่วงเวลาเล็กน้อย

print(f"บันทึกข้อมูลเรียบร้อยแล้ว ✅ ที่ {save_path}")
